In [1]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/MEGPypes/notebooks
/Users/peli/Projects/Repositories/MEGPypes
Working Dir Base: /Users/peli/Projects/Repositories/MEGPypes


In [ ]:
import yaml
import time
from bids.layout import BIDSLayout
# import
from src.pipelines.init_preproc import create_initial_preprocessing

loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


First we need to grab data from our dataset.
We assume the data operate within a bids compliant data structure
https://peerherholz.github.io/workshop_weizmann/nipype/notebooks/basic_data_input_bids.html



In [4]:
# do some data inspection using pybids?
layout = BIDSLayout("data/ds006629/")
print(layout)
# subjects
subjects = layout.get_subjects()
print(f"Subjects: {subjects}")
# datatypes
bidstypes = layout.get_datatypes()
print(f"Data types: {bidstypes}")
# suffixes
print(layout.get_suffixes(datatype='func'))
# see tasks
layout.get_tasks()
# see dataset description
layout.get_dataset_description()
#

# see data metadata


BIDS Layout: ...itories/MEGPypes/data/ds006629 | Subjects: 19 | Sessions: 0 | Runs: 19
Subjects: ['01', '02', '04', '05', '06', '07', '08', '09', '10', '11', '12', '14', '15', '16', '17', '18', '19', '20', '21']
Data types: ['meg']
[]


{'Name': 'SINGSING',
 'BIDSVersion': '1.7.0',
 'License': 'CC0',
 'DatasetType': 'raw',
 'Authors': ['Valerie Chanoine',
  'Jean-Michel Badier',
  'Mireille Besson',
  'Talya Inbar'],
 'Acknowledgements': 'MEG data acquisition was performed in the MEG Centre (Timone Hospital, Marseille, France)',
 'Funding': ['This research has been supported by funding from the Institute of Convergence ILCB (France 2030, ANR-16-CONV-0002) and the Excellence Initiative of Aix-Marseille University A*MIDEX (ANR-11-IDEX-0001-02)'],
 'ReferencesAndLinks': ['a data paper',
  'a resource to be cited when using the data'],
 'DatasetDOI': 'doi:10.18112/openneuro.ds006629.v1.0.1',
 'GeneratedBy': [{'Name': 'MNE-BIDS',
   'Version': '0.14',
   'Description': 'MNE-BIDS is a Python package that allows you to read and write BIDS-compatible datasets with the help of MNE-Python.'}],
 'SourceDatasets': [{'DOI': 'doi:10.18112/openneuro.ds006629.v1.0.0',
   'URL': 'https://openneuro.org/datasets/ds006629',
   'Version':

In [ ]:
# Load configs
import os
import yaml
from nipype import config as nconfig

config_path = "config/config_ds006629.yaml"
with open(config_path, "r") as yamlfile:
    config = yaml.load(yamlfile, Loader=yaml.FullLoader)

paths_config = config["paths"]
proc_config = config["processing"]
wf_config = config["workflow"]

# Configure Nipype logging (applies to all subprocesses)
nconfig.update_config({
    'logging': {
        'log_directory': os.path.join(paths_config["workdir"], 'logs'),
        'log_to_file': True,
        'interface_level': 'info',
        'workflow_level': 'info',
    },
    'execution': {
        'crashdump_dir': os.path.abspath('crashes'),
        'remove_unnecessary_outputs': False,
    }
})

# Create workflow
wf = create_initial_preprocessing(
    basedir=paths_config["basedir"],
    workdir=paths_config["workdir"],
    output_dir=paths_config["outputdir"],
    subject_list=paths_config["subjects"],  # ← Now actually used!
    crop_params=proc_config["crop"],
    filter_params=proc_config["filter"],
    gradcomp_params=proc_config["gradcomp"]
)

# visualize workflow graph
wf.write_graph(graph2use='colored', simple_form=True)
print(f"Workflow graph saved to: {wf.base_dir}/megpreproc/graph.png")

# Run workflow
n_workers = wf_config.get("n_workers", max(1, os.cpu_count() - 2))
print(f"Running with {n_workers} workers")

result = wf.run(
    plugin=wf_config["plugin"],
    plugin_args={"n_procs": n_workers}
)

raw_dir /Users/peli/Projects/Repositories/MEGPypes/data/ds006629
260224-19:50:36,11 nipype.workflow INFO:
	 Generated workflow graph: workdir/megpreproc/graph.png (graph2use=colored, simple_form=True).
Workflow graph saved to: workdir/megpreproc/graph.png
Running with 8 workers
260224-19:50:36,19 nipype.workflow INFO:
	 Workflow megpreproc settings: ['check', 'execution', 'logging', 'monitoring']
260224-19:50:36,23 nipype.workflow INFO:
	 Running serially.
260224-19:50:36,23 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.selectfiles" in "/Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-01/selectfiles".
260224-19:50:36,25 nipype.workflow INFO:
	 [Node] Executing "selectfiles" <nipype.interfaces.io.SelectFiles>
260224-19:50:36,26 nipype.workflow INFO:
	 [Node] Finished "selectfiles", elapsed time 0.000406s.
260224-19:50:36,28 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.selectfiles" in "/Users/peli/Projects/Repositories/MEGPypes/workdir/megpre

2026-02-24 19:50:36,033 [INFO] src.interfaces.initpreproc: Initial preproc: /Users/peli/Projects/Repositories/MEGPypes/data/ds006629/sub-01/meg/sub-01_task-MMNHCS_run-0_meg.fif


Opening raw data file /Users/peli/Projects/Repositories/MEGPypes/data/ds006629/sub-01/meg/sub-01_task-MMNHCS_run-0_meg.fif...
    Read a total of 1 projection items:
        axial-Raw-0.000-900.800-PCA-01 (1 x 229)  idle
    Range : 0 ... 225199 =      0.000 ...   900.796 secs
Ready.
Reading 0 ... 225199  =      0.000 ...   900.796 secs...
Finding events on: TRIGGER, RESPONSE
1215 events found on stim channel TRIGGER
Event IDs: [  2   4   6   8 254]
Trigger channel RESPONSE has a non-zero initial value of 64 (consider using initial_event=True to detect this event)
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passba

2026-02-24 19:50:37,491 [WARNING] src.interfaces.initpreproc: No gradcomp matrices available


Writing /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-01/initial_preproc/preproc_raw.fif
Closing /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-01/initial_preproc/preproc_raw.fif
[done]


...

Opening raw data file /Users/peli/Projects/Repositories/MEGPypes/data/ds006629/sub-02/meg/sub-02_task-MMNHCS_run-0_meg.fif...
    Read a total of 1 projection items:
        axial-Raw-0.000-813.092-PCA-01 (1 x 231)  idle
    Range : 0 ... 203272 =      0.000 ...   813.088 secs
Ready.
Reading 0 ... 203272  =      0.000 ...   813.088 secs...
Finding events on: TRIGGER, RESPONSE
1215 events found on stim channel TRIGGER
Event IDs: [  2   4   6   8 254]
Trigger channel RESPONSE has a non-zero initial value of 64 (consider using initial_event=True to detect this event)
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passba

2026-02-24 19:50:38,690 [WARNING] src.interfaces.initpreproc: No gradcomp matrices available


Writing /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-02/initial_preproc/preproc_raw.fif
Closing /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-02/initial_preproc/preproc_raw.fif
[done]


2026-02-24 19:50:38,856 [INFO] src.interfaces.initpreproc: Saved: /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-02/initial_preproc/preproc_raw.fif


260224-19:50:38,857 nipype.workflow INFO:
	 [Node] Finished "initial_preproc", elapsed time 1.13781s.
260224-19:50:38,858 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.datasink" in "/Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-01/datasink".
260224-19:50:38,860 nipype.workflow INFO:
	 [Node] Executing "datasink" <nipype.interfaces.io.DataSink>
260224-19:50:38,861 nipype.workflow INFO:
	 [Node] Finished "datasink", elapsed time 0.000626s.
260224-19:50:38,862 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.datasink" in "/Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-02/datasink".
260224-19:50:38,863 nipype.workflow INFO:
	 [Node] Executing "datasink" <nipype.interfaces.io.DataSink>
260224-19:50:38,865 nipype.workflow INFO:
	 [Node] Finished "datasink", elapsed time 0.0006s.
